# 01 - Data Cleaning

Load the four raw sales files (hourly, daily, weekly, monthly), standardize dates, reshape each to long format, validate consistency across granularities, and load everything into a SQLite database for the SQL analysis step.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
from pathlib import Path

DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

CATEGORY_COLS = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]

## Load raw files

Each file uses a different date format, so they are parsed explicitly rather than relying on inference.

In [2]:
monthly = pd.read_csv(DATA_DIR / "salesmonthly.csv")
weekly = pd.read_csv(DATA_DIR / "salesweekly.csv")
daily = pd.read_csv(DATA_DIR / "salesdaily.csv")
hourly = pd.read_csv(DATA_DIR / "saleshourly.csv")

monthly["datum"] = pd.to_datetime(monthly["datum"], format="%Y-%m-%d")
weekly["datum"] = pd.to_datetime(weekly["datum"], format="%m/%d/%Y")
daily["datum"] = pd.to_datetime(daily["datum"], format="%m/%d/%Y")
hourly["datum"] = pd.to_datetime(hourly["datum"], format="%m/%d/%Y %H:%M")

for name, df in [("monthly", monthly), ("weekly", weekly), ("daily", daily), ("hourly", hourly)]:
    print(f"{name:8s} shape={df.shape}  range=({df['datum'].min().date()} -> {df['datum'].max().date()})")

monthly  shape=(70, 9)  range=(2014-01-31 -> 2019-10-31)
weekly   shape=(302, 9)  range=(2014-01-05 -> 2019-10-13)
daily    shape=(2106, 13)  range=(2014-01-02 -> 2019-10-08)
hourly   shape=(50532, 13)  range=(2014-01-02 -> 2019-10-08)


## Missing values and duplicate dates

In [3]:
for name, df in [("monthly", monthly), ("weekly", weekly), ("daily", daily), ("hourly", hourly)]:
    n_missing = df[CATEGORY_COLS].isna().sum().sum()
    n_negative = (df[CATEGORY_COLS] < 0).sum().sum()
    n_dupe_dates = df["datum"].duplicated().sum()
    print(f"{name:8s} missing={n_missing}  negative={n_negative}  duplicate_dates={n_dupe_dates}")

monthly  missing=0  negative=0  duplicate_dates=0
weekly   missing=0  negative=0  duplicate_dates=0
daily    missing=0  negative=0  duplicate_dates=0
hourly   missing=0  negative=0  duplicate_dates=0


## Reshape to long format

Each granularity is melted from one-column-per-category to `(sale_date, category_code, units_sold)`, which is the shape everything downstream (SQL, EDA, forecasting) expects.

In [4]:
def to_long(df, granularity):
    long_df = df.melt(
        id_vars="datum", value_vars=CATEGORY_COLS,
        var_name="category_code", value_name="units_sold"
    )
    long_df = long_df.rename(columns={"datum": "sale_date"})
    long_df["granularity"] = granularity
    long_df["units_sold"] = long_df["units_sold"].clip(lower=0)
    return long_df.sort_values(["category_code", "sale_date"]).reset_index(drop=True)

monthly_long = to_long(monthly, "monthly")
weekly_long = to_long(weekly, "weekly")
daily_long = to_long(daily, "daily")
hourly_long = to_long(hourly, "hourly")

monthly_long.head()

,sale_date,category_code,units_sold,granularity
0,2014-01-31,M01AB,127.69,monthly
1,2014-02-28,M01AB,133.32,monthly
2,2014-03-31,M01AB,137.44,monthly
3,2014-04-30,M01AB,113.10,monthly
4,2014-05-31,M01AB,101.79,monthly


## Validate: daily rolled up to monthly should match `salesmonthly.csv`

Sanity check that the granularities agree with each other before trusting any of them.

In [5]:
daily_check = daily_long.copy()
daily_check["month"] = daily_check["sale_date"].dt.to_period("M")

rollup = (
    daily_check.groupby(["month", "category_code"])["units_sold"]
    .sum()
    .reset_index()
)

monthly_check = monthly_long.copy()
monthly_check["month"] = monthly_check["sale_date"].dt.to_period("M")

merged = rollup.merge(
    monthly_check,
    on=["month", "category_code"],
    suffixes=("_rollup", "_reported"),
)
merged["abs_diff"] = (merged["units_sold_rollup"] - merged["units_sold_reported"]).abs()
merged["abs_diff"].describe()

count    5.600000e+02
mean     8.843024e+00
std      7.375310e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      4.618528e-14
max      1.439360e+03
Name: abs_diff, dtype: float64

The last calendar month in the daily/hourly files may be partial (data collection cutoff), so expect the largest mismatch there — check it explicitly rather than assuming.

In [6]:
merged.sort_values("abs_diff", ascending=False).head(10)

,month,category_code,units_sold_rollup,sale_date,units_sold_reported,granularity,abs_diff
291,2017-01,N02BE,1439.360000,2017-01-31,0.000,monthly,1439.360000
75,2014-10,N02BE,1073.136875,2014-10-31,1856.815,monthly,783.678125
292,2017-01,N05B,443.750000,2017-01-31,1.000,monthly,442.750000
294,2017-01,R03,196.406250,2017-01-31,0.000,monthly,196.406250
290,2017-01,N02BA,191.270833,2017-01-31,0.000,monthly,191.270833
288,2017-01,M01AB,181.902083,2017-01-31,0.000,monthly,181.902083
289,2017-01,M01AE,150.991250,2017-01-31,0.000,monthly,150.991250
299,2017-02,N02BE,670.875833,2017-02-28,526.350,monthly,144.525833
73,2014-10,M01AE,108.751312,2014-10-31,185.241,monthly,76.489688
251,2016-08,N02BE,686.383333,2016-08-31,753.050,monthly,66.666667


**Finding: `salesmonthly.csv` has a missing/zeroed month.** January 2017 is reported as `0.0` for 7 of the 8 categories in the raw monthly file, while the daily file shows real sales that month (e.g. N02BE: ~1,439 units from the daily rollup vs. 0 reported). This is a genuine gap in the source file, not a bug in the rollup logic — confirmed directly against the raw CSV.

This matters downstream: decomposition, ACF/PACF, and SARIMA in later notebooks should either (a) use the daily-rollup-derived monthly series instead of the raw monthly file, or (b) explicitly impute January 2017 before differencing/fitting, since an unflagged zero will look like a demand collapse and distort seasonality estimates. Carry this decision into `03_statistical_analysis`.

Patch it here rather than downstream: replace the January 2017 monthly values with the daily-rollup total for that month, so every later notebook inherits a corrected series instead of each one having to remember the workaround.

In [7]:
jan_2017_fix = rollup[rollup["month"] == pd.Period("2017-01", freq="M")][["category_code", "units_sold"]]
jan_2017_fix = jan_2017_fix.set_index("category_code")["units_sold"]

target_rows = monthly_long["sale_date"] == pd.Timestamp("2017-01-31")
for code, value in jan_2017_fix.items():
    monthly_long.loc[target_rows & (monthly_long["category_code"] == code), "units_sold"] = value

monthly_long[target_rows]

,sale_date,category_code,units_sold,granularity
36,2017-01-31,M01AB,181.902083,monthly
106,2017-01-31,M01AE,150.991250,monthly
176,2017-01-31,N02BA,191.270833,monthly
246,2017-01-31,N02BE,1439.360000,monthly
316,2017-01-31,N05B,443.750000,monthly
386,2017-01-31,N05C,34.166667,monthly
456,2017-01-31,R03,196.406250,monthly
526,2017-01-31,R06,61.750000,monthly


## Save processed long-format files

In [8]:
monthly_long.to_csv(PROCESSED_DIR / "monthly_long.csv", index=False)
weekly_long.to_csv(PROCESSED_DIR / "weekly_long.csv", index=False)
daily_long.to_csv(PROCESSED_DIR / "daily_long.csv", index=False)
hourly_long.to_csv(PROCESSED_DIR / "hourly_long.csv", index=False)

## Load into SQLite

Builds `dim_category` + `fact_sales` from `sql/schema.sql`, then loads all four granularities into `fact_sales` (tagged by the `granularity` column) so `sql/sales_analysis.sql` has real data to query against.

In [9]:
db_path = PROCESSED_DIR / "pharma_sales.db"
conn = sqlite3.connect(db_path)

with open("../sql/schema.sql") as f:
    conn.executescript(f.read())

all_long = pd.concat([monthly_long, weekly_long, daily_long, hourly_long], ignore_index=True)
all_long["sale_date"] = all_long["sale_date"].astype(str)

all_long[["sale_date", "category_code", "granularity", "units_sold"]].to_sql(
    "fact_sales", conn, if_exists="append", index=False
)
conn.commit()

## Sanity check

In [10]:
pd.read_sql(
    """
    SELECT f.granularity, d.category_name, COUNT(*) AS n_rows, ROUND(SUM(f.units_sold), 1) AS total_units
    FROM fact_sales f
    JOIN dim_category d ON f.category_code = d.category_code
    GROUP BY f.granularity, d.category_name
    ORDER BY f.granularity, total_units DESC
    """,
    conn,
)

,granularity,category_name,n_rows,total_units
0,daily,Analgesics - pyrazolones and anilides,2106,63005.4
1,daily,Anxiolytics,2106,18645.7
2,daily,Anti-asthmatic and COPD drugs,2106,11608.8
3,daily,Anti-inflammatory (acetic acid derivatives),2106,10600.9
4,daily,Anti-inflammatory (propionic acid derivatives),2106,8204.6
5,daily,Analgesics - salicylic acid derivatives,2106,8172.2
6,daily,Antihistamines,2106,6107.8
7,daily,Hypnotics and sedatives,2106,1250.0
8,hourly,Analgesics - pyrazolones and anilides,50532,63005.4
9,hourly,Anxiolytics,50532,18645.7


In [11]:
conn.close()